**Pre Trained ResNet34**

**Libraries**

In [22]:
import sys
from pathlib import Path

cloned_rep_path=r'C:\Users\laika\OneDrive\Documents\Capstone I All Files\Capstone-DATA4381'
sys.path.insert(0,cloned_rep_path)

from capstone_data.image_loader import load_dataset, split_dataset, load_arrays

from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models import resnet34, ResNet34_Weights


**Load Dataset**

In [ ]:

aug_datasets=['aug_data_v1', 'aug_data_v2', 'aug_data_v3']
base_path=Path(r'c:\Users\laika\OneDrive\Documents\Capstone I All Files\Capstone-DATA4381\data\augmented_data')

for name in aug_datasets:
    folder_path = base_path / name
    
    # Loop through every image in the subfolders
    
    for img_path in folder_path.glob("**/*.*"):
        
        if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            
            with Image.open(img_path) as img:
                
                # Apply .convert("L") and .resize()
                
                img = img.convert("L")
                
                img = img.resize((48, 48))
                
                img.save(img_path)
                

aug_dict={}

for name in aug_datasets:
    
    
    aug_load=load_dataset(name)
    
    X,y=load_arrays(aug_load)
    
    X_train, y_train, X_val, y_val, X_test, y_test = split_dataset(
            X, y, 
            train_ratio=0.80, 
            val_ratio=0.00,
            test_ratio=0.20, 
            seed=42
        )    

    # Convert Numpy arrays to PyTorch tensors & Reformat for ResNet34
    
    X_train_tensor=torch.tensor(X_train, dtype=torch.float32).unsqueeze(1).repeat(1, 3, 1, 1) #
    X_test_tensor=torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).repeat(1, 3, 1, 1)
    y_train_tensor=torch.tensor(y_train, dtype=torch.long) # torch.long to convert to interger labels
    y_test_tensor=torch.tensor(y_test, dtype=torch.long)
    
    # Create Pytorch Dataloaders
    # look at 32 images at a time
    train_load=DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
    test_load=DataLoader(TensorDataset(X_test_tensor,y_test_tensor), batch_size=32, shuffle=False)
    
    # Store each version
    
    aug_dict[name]={
        'train': train_load,
        'test': test_load
    }

Loading images nerds:   0%|          | 0/14019 [00:00<?, ?it/s]

Loading images nerds:   0%|          | 0/12069 [00:00<?, ?it/s]

Loading images nerds:   0%|          | 0/12064 [00:00<?, ?it/s]

**Modify Resnet34 model**

In [17]:
# Modify ResNet34 for 3 classes & 48x48  images

def modify_resnet34(num_classes=3):
    
    model=resnet34(weights=None) # No weights, since default weights expect 224x224 colored images
    
    # Modify the first layer for 48,48 images
    model.conv1=nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    
    # Remove maxpool layer, to avoid shrinking image features too much
    model.maxpool=nn.Identity()
    
    # Final layer
    model.fc=nn.Linear(model.fc.in_features,num_classes)
    
    return model

In [ ]:
# Initialize the model

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=modify_resnet34(num_classes=3).to(device)

In [20]:
# Test Batch

test_batch=aug_dict['aug_data_v1']['train']
images,labels=next(iter(test_batch))
output=model(images)

print(f'Input batch shape: {images.shape}')
print(f'Output shape: {output.shape}')

Input batch shape: torch.Size([32, 3, 48, 48])
Output shape: torch.Size([32, 3])


**Train the Model**

In [23]:

class ResNet34:
    def __init__(self, aug_dict, num_classes=3, lr=0.001, device=None):
        self.aug_dict=aug_dict
        self.num_classes=num_classes
        self.lr=lr
        self.device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.trained_versions={}
    
    def train_model(self,num_epochs=3):
        
        # Iterate through all versions
        for name, data in self.aug_dict.items():
            print(f'Training: {name}')
            
            model=modify_resnet34(num_classes=self.num_classes).to(self.device)
            
            # Loss function
            criterion=nn.CrossEntropyLoss()
            
            # Optimizer
            optimizer=optim.Adam(model.parameters(), lr=self.lr)
            
            train_data=data['train']
            
            # Epochs
            for epoch in range(num_epochs):
                model.train()
                total_loss=0.0 #total loss per batch
                
                for images,labels in train_data:
                    
                    # Ensure images and labels are both on either CPU or GPU
                    images,labels=images.to(self.device),labels.to(self.device)
                    
                    # Add up gradients from previous step
                    optimizer.zero_grad()
                    
                    # Forward pass
                    outputs=model(images)
                    
                    # Compute loss (Compare model's predions with the true labels)
                    loss=criterion(outputs, labels)

                    # Backpropagation
                    loss.backward()
                    
                    # Update weights
                    optimizer.step()
                    
                    total_loss+=loss.item()
                
                avg_loss=total_loss/len(train_data)
                print(f'Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss}')
                
            # Store model version
            self.trained_versions[name]=model
            print(f'Training completed for {name}')

In [ ]:
train=ResNet34(aug_dict, num_classes=3, lr=0.001, device=device)
trained_models=train.train_model(num_epochs=3)

Training: aug_data_v1
